# 03 — CTR-GCN Supervised Fine-tuning

Fine-tune CTR-GCN on labeled tennis stroke data (728 train / 91 val).
Optionally loads AimCLR pretrained backbone from Phase 4.

**Target metrics:**
- Val Top-1 accuracy ≥ 90% (4 valid stroke classes)
- Per-class F1 ≥ 0.85
- FPR (invalid misclassified as stroke) ≤ 10%

**Input convention:** `(N, 3, 45, 17, 1)` — batch × (x_model, y_model, conf) × frames × joints × person

**Class mapping:** 0=forehand, 1=backhand, 2=serve, 3=volley, 4=invalid

## Cell 1 — Mount Drive & Clone Repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys, os

REPO_URL = 'https://github.com/michaelljx/tennis-vision-training.git'
REPO_DIR = '/content/repo'

if os.path.isdir(os.path.join(REPO_DIR, '.git')):
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
    print('Pulled latest changes')
else:
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
    print('Cloned repo')

sys.path.insert(0, REPO_DIR)
print(f'Repo: {REPO_DIR}')

## Cell 2 — Config & GPU Check

In [ ]:
import torch
import numpy as np

# ── Paths ──────────────────────────────────────────────────────────────────
DRIVE_DATA   = '/content/drive/MyDrive/Tennis-CTR-GCN/data'
DRIVE_MODELS = '/content/drive/MyDrive/Tennis-CTR-GCN/models'

TRAIN_NPZ    = f'{DRIVE_DATA}/train.npz'
VAL_NPZ      = f'{DRIVE_DATA}/val.npz'
PRETRAIN_PTH = f'{DRIVE_MODELS}/aimclr_pretrain.pth'   # may not exist
BEST_PTH     = f'{DRIVE_MODELS}/ctrgcn_best.pth'

import os
os.makedirs(DRIVE_MODELS, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch: {torch.__version__}')
print(f'Device:  {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU:     {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU. Enable GPU in Runtime > Change runtime type.')

USE_PRETRAIN = os.path.exists(PRETRAIN_PTH)
print(f'\nAimCLR pretrain checkpoint: {"FOUND ✓" if USE_PRETRAIN else "NOT FOUND — using random init"}')

## Cell 3 — Load Data & Check Distribution

In [ ]:
train_data = np.load(TRAIN_NPZ, allow_pickle=True)
val_data   = np.load(VAL_NPZ,   allow_pickle=True)

X_train, y_train = train_data['X'], train_data['y']
X_val,   y_val   = val_data['X'],   val_data['y']

CLASS_NAMES = ['forehand', 'backhand', 'serve', 'volley', 'invalid']
N_CLASSES   = len(CLASS_NAMES)

print(f'Train: {X_train.shape}  labels: {y_train.shape}')
print(f'Val:   {X_val.shape}  labels: {y_val.shape}')
print()

from collections import Counter
train_dist = Counter(y_train.tolist())
val_dist   = Counter(y_val.tolist())

print('Class distribution:')
print(f'  {"Class":<12} {"Train":>8} {"Val":>6}')
for i, name in enumerate(CLASS_NAMES):
    print(f'  {name:<12} {train_dist[i]:>8} {val_dist[i]:>6}')

# Compute class weights for Focal Loss (inverse frequency)
import torch
counts = torch.tensor([train_dist[i] for i in range(N_CLASSES)], dtype=torch.float)
class_weights = (counts.sum() / (N_CLASSES * counts)).to(DEVICE)
print(f'\nClass weights (inv-freq): {[f"{w:.2f}" for w in class_weights.cpu()]}') 

## Cell 4 — Focal Loss

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class FocalLoss(nn.Module):
    """
    Focal Loss with class weighting.
    Reduces relative loss for well-classified examples, focusing on hard cases.
    FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)
    """
    def __init__(self, gamma: float = 2.0, weight=None):
        super().__init__()
        self.gamma  = gamma
        self.weight = weight   # (num_class,) tensor

    def forward(self, logits, targets):
        # logits: (N, C), targets: (N,)
        ce   = F.cross_entropy(logits, targets, weight=self.weight, reduction='none')  # (N,)
        pt   = torch.exp(-ce)        # probability of correct class
        loss = ((1 - pt) ** self.gamma) * ce
        return loss.mean()

# Verify
fl = FocalLoss(gamma=2.0, weight=class_weights)
dummy_logits = torch.randn(4, N_CLASSES).to(DEVICE)
dummy_labels = torch.randint(0, N_CLASSES, (4,)).to(DEVICE)
loss_val = fl(dummy_logits, dummy_labels)
print(f'FocalLoss test: {loss_val.item():.4f} ✓')

## Cell 5 — Dataset & DataLoader

In [ ]:
from torch.utils.data import DataLoader
from aimclr.dataset import LabeledSkeletonDataset
from aimclr.augmentations import compose_augmentation

BATCH_SIZE = 64

train_dataset = LabeledSkeletonDataset(TRAIN_NPZ, augment_fn=compose_augmentation)
val_dataset   = LabeledSkeletonDataset(VAL_NPZ,   augment_fn=None)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, drop_last=False, pin_memory=(DEVICE=='cuda'))
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=(DEVICE=='cuda'))

print(f'Train: {len(train_dataset)} samples, {len(train_loader)} batches')
print(f'Val:   {len(val_dataset)} samples,  {len(val_loader)} batches')

# Verify shape
x_b, y_b = next(iter(train_loader))
print(f'Batch x: {x_b.shape}  y: {y_b.shape}')  # (B, 3, 45, 17), (B,)

## Cell 6 — Model & Optimizer

In [ ]:
from ctrgcn.model import CTRGCN

model = CTRGCN(num_class=N_CLASSES, base_channels=64, adaptive=True).to(DEVICE)

# Load AimCLR pretrained backbone if available
if USE_PRETRAIN:
    state = torch.load(PRETRAIN_PTH, map_location=DEVICE)
    # Load with strict=False — FC head weights will be missing (initialized fresh)
    missing, unexpected = model.load_state_dict(state, strict=False)
    missing_non_fc = [k for k in missing if not k.startswith('fc.')]
    print(f'Loaded pretrain weights.')
    print(f'  Missing (non-FC): {missing_non_fc or "none"}')
    print(f'  Unexpected: {unexpected or "none"}')
else:
    print('Random initialization.')

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable params: {n_params:,}')

# SGD + Nesterov (standard for action recognition)
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.1,
    momentum=0.9,
    nesterov=True,
    weight_decay=4e-4,
)

EPOCHS = 80
WARMUP = 5   # linear warmup epochs

def lr_lambda(epoch):
    if epoch < WARMUP:
        return (epoch + 1) / WARMUP
    # Cosine decay after warmup
    import math
    progress = (epoch - WARMUP) / (EPOCHS - WARMUP)
    return 0.5 * (1 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
criterion = FocalLoss(gamma=2.0, weight=class_weights)

print(f'Optimizer: SGD lr=0.1, momentum=0.9, nesterov, wd=4e-4')
print(f'Scheduler: Linear warmup ({WARMUP} ep) + Cosine decay ({EPOCHS} ep total)')

## Cell 7 — Training Loop

In [ ]:
from sklearn.metrics import f1_score, confusion_matrix

history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_f1': []}
best_val_acc = 0.0
patience_counter = 0
PATIENCE = 15

def run_epoch(loader, train=True):
    model.train(train)
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for x, y in loader:
            x = x.unsqueeze(-1).to(DEVICE)  # (N, 3, 45, 17) → (N, 3, 45, 17, 1)
            y = y.to(DEVICE)

            logits = model(x)
            loss   = criterion(logits, y)

            if train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            preds = logits.argmax(dim=1)
            correct      += (preds == y).sum().item()
            total        += y.size(0)
            total_loss   += loss.item() * y.size(0)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(y.cpu().tolist())

    avg_loss = total_loss / total
    acc      = correct / total
    f1       = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return avg_loss, acc, f1, all_preds, all_labels


print(f'Training for {EPOCHS} epochs  (early stop patience={PATIENCE})\n')
print(f'{"Epoch":>5}  {"TrainLoss":>10}  {"ValLoss":>8}  {"ValAcc":>7}  {"ValF1":>6}  {"LR":>8}')
print('─' * 60)

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc, tr_f1, _, _ = run_epoch(train_loader, train=True)
    va_loss, va_acc, va_f1, va_preds, va_labels = run_epoch(val_loader, train=False)
    scheduler.step()

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(va_loss)
    history['val_acc'].append(va_acc)
    history['val_f1'].append(va_f1)

    lr_now = scheduler.get_last_lr()[0]

    if epoch % 5 == 0 or epoch == 1:
        print(f'{epoch:>5}  {tr_loss:>10.4f}  {va_loss:>8.4f}  {va_acc:>7.3f}  {va_f1:>6.3f}  {lr_now:>8.1e}')

    # Save best checkpoint
    if va_acc > best_val_acc:
        best_val_acc = va_acc
        best_val_f1  = va_f1
        best_preds   = va_preds
        best_labels  = va_labels
        patience_counter = 0
        torch.save(model.state_dict(), BEST_PTH)
        print(f'       ↑ New best: acc={va_acc:.3f}  f1={va_f1:.3f}  → saved')
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'\nEarly stop at epoch {epoch} (patience={PATIENCE})')
            break

print('\nTraining complete.')
print(f'Best val acc: {best_val_acc:.4f}  Best val F1: {best_val_f1:.4f}')

## Cell 8 — Evaluation: Metrics, Confusion Matrix, FPR

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix

# Reload best checkpoint for final eval
model.load_state_dict(torch.load(BEST_PTH, map_location=DEVICE))
_, final_acc, _, final_preds, final_labels = run_epoch(val_loader, train=False)

print(f'=== Final Val Metrics (best checkpoint) ===')
print(f'Top-1 Accuracy: {final_acc:.4f}  ({final_acc*100:.1f}%)')
print()
print(classification_report(final_labels, final_preds, target_names=CLASS_NAMES, zero_division=0))

# FPR: invalid class being misclassified as a valid stroke
inv_idx    = CLASS_NAMES.index('invalid')
inv_mask   = np.array(final_labels) == inv_idx
inv_preds  = np.array(final_preds)[inv_mask]
fpr        = (inv_preds != inv_idx).sum() / max(1, inv_mask.sum())
print(f'FPR (invalid→stroke): {fpr:.3f}  ({fpr*100:.1f}%)  [target ≤ 10%]')

# Go/No-Go gate
print()
f1_per_class = f1_score(final_labels, final_preds, average=None, zero_division=0)
min_f1 = min(f1_per_class[:4])  # 4 valid stroke classes only
pass_acc = final_acc >= 0.90
pass_f1  = min_f1   >= 0.85
pass_fpr = fpr      <= 0.10

print('Go/No-Go Gate:')
print(f'  Val acc ≥ 90%:           {"✓ PASS" if pass_acc else "✗ FAIL"}  ({final_acc*100:.1f}%)')
print(f'  Min per-class F1 ≥ 0.85: {"✓ PASS" if pass_f1  else "✗ FAIL"}  ({min_f1:.3f})')
print(f'  FPR ≤ 10%:               {"✓ PASS" if pass_fpr else "✗ FAIL"}  ({fpr*100:.1f}%)')
if pass_acc and pass_f1 and pass_fpr:
    print('\n✓ GO — proceed to Phase 6 (model export)')
else:
    print('\n✗ NO-GO — collect more data and retrain')

## Cell 9 — Confusion Matrix & Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Confusion matrix
cm = confusion_matrix(final_labels, final_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
im = axes[0].imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
axes[0].set_xticks(range(N_CLASSES)); axes[0].set_xticklabels(CLASS_NAMES, rotation=30, ha='right')
axes[0].set_yticks(range(N_CLASSES)); axes[0].set_yticklabels(CLASS_NAMES)
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')
axes[0].set_title('Confusion Matrix (row-normalized)')
for i in range(N_CLASSES):
    for j in range(N_CLASSES):
        axes[0].text(j, i, f'{cm_norm[i,j]:.2f}', ha='center', va='center',
                     color='white' if cm_norm[i,j] > 0.5 else 'black', fontsize=9)
plt.colorbar(im, ax=axes[0])

# Loss curves
epochs_range = range(1, len(history['train_loss']) + 1)
axes[1].plot(epochs_range, history['train_loss'], label='Train Loss')
axes[1].plot(epochs_range, history['val_loss'],   label='Val Loss')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Focal Loss')
axes[1].set_title('Loss Curves')
axes[1].legend(); axes[1].grid(alpha=0.3)

# Accuracy & F1
axes[2].plot(epochs_range, history['val_acc'], label='Val Acc', color='green')
axes[2].plot(epochs_range, history['val_f1'],  label='Val F1',  color='orange')
axes[2].axhline(0.90, color='green',  linestyle='--', alpha=0.5, label='Target acc 90%')
axes[2].axhline(0.85, color='orange', linestyle='--', alpha=0.5, label='Target F1 85%')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Score')
axes[2].set_title('Val Accuracy & Macro F1')
axes[2].legend(); axes[2].grid(alpha=0.3)

plt.tight_layout()
plot_path = f'{DRIVE_MODELS}/ctrgcn_training.png'
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {plot_path}')

## Cell 10 — Save Final Checkpoint Summary

In [ ]:
import json

# Save metadata alongside the checkpoint
meta = {
    'val_acc':       round(float(final_acc), 4),
    'val_f1_macro':  round(float(best_val_f1), 4),
    'val_f1_per_class': {CLASS_NAMES[i]: round(float(f1_per_class[i]), 4) for i in range(N_CLASSES)},
    'fpr_invalid':   round(float(fpr), 4),
    'epochs_trained': len(history['train_loss']),
    'pretrain_used':  USE_PRETRAIN,
    'go_nogo':        'GO' if (pass_acc and pass_f1 and pass_fpr) else 'NO-GO',
}
meta_path = f'{DRIVE_MODELS}/ctrgcn_best_meta.json'
with open(meta_path, 'w') as f:
    json.dump(meta, f, indent=2)

print('Checkpoint metadata:')
print(json.dumps(meta, indent=2))
print(f'\nSaved: {meta_path}')
print(f'Model: {BEST_PTH}')
size_mb = os.path.getsize(BEST_PTH) / 1e6
print(f'Size:  {size_mb:.1f} MB')
print()
if meta['go_nogo'] == 'GO':
    print('✓ Ready for Phase 6: export PyTorch → ONNX → CoreML INT8')
    print('  Next: run export/pytorch_to_onnx.py, then export/onnx_to_coreml.py')
else:
    print('Metrics below target. Options:')
    print('  1. Collect more data (especially volley/serve) and retrain')
    print('  2. Try lower base_channels=32 to reduce overfitting')
    print('  3. Check that AimCLR pretrain ran successfully before retraining')